# 08 · NGSIM stop-and-go — Unified clustering

Combines the personalized components of **acceleration, spacing, and speed**
(notebooks 05–07) into a single feature vector per vehicle and clusters the
vehicles jointly (top-5 PCA scores, k-means with k = 3). This is the joint
"behavioral fingerprint" view of the NGSIM drivers.

**Figure produced:** `NGSIM_cluster.png`.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "bhd").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.interpolate import interp1d
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from bhd import DATA_DIR, FIG_DIR, set_style, load_ngsim_stop_to_go

set_style()

## 1. Load, filter, and align all three signals

In [ ]:
df = load_ngsim_stop_to_go(DATA_DIR / "ngsim" / "stop_to_go.csv")

valid_ids = [tid for tid in df["trajectory_id"].unique() if (df.loc[df["trajectory_id"] == tid, "spacing"] >= 0).all()]
df = df[df["trajectory_id"].isin(valid_ids)]
min_len = min(len(df[df["trajectory_id"] == tid]) for tid in valid_ids)
print(f"{len(valid_ids)} trajectories, resampled to {min_len} points")


def resample_series(series):
    return interp1d(np.linspace(0, 1, len(series)), series)(np.linspace(0, 1, min_len))


acc_list, spc_list, spd_list = [], [], []
for tid in valid_ids:
    group = df[df["trajectory_id"] == tid]
    acc_list.append(resample_series(group["follower_acc"].values))
    spc_list.append(resample_series(group["spacing"].values))
    spd_list.append(resample_series(group["follower_speed"].values))

A = np.stack(acc_list, axis=1)  # (T, N)
S = np.stack(spc_list, axis=1)
V = np.stack(spd_list, axis=1)

## 2. Two-shot decomposition of each signal

In [ ]:
def decompose_two_shot(X, force_positive=False):
    pca_shared = PCA(n_components=1)
    C = pca_shared.fit(X.T).components_.T
    Psi = C.T @ X
    X_shared = C @ Psi
    personalized = []
    for i in range(X.shape[1]):
        R = (X[:, i] - X_shared[:, i]).reshape(-1, 1)
        pca_personal = PCA(n_components=1)
        Phi = pca_personal.fit_transform(R)
        R_hat = pca_personal.inverse_transform(Phi)
        x_hat = X_shared[:, i] + R_hat.flatten()
        if force_positive:
            x_hat = np.clip(x_hat, 0, None)
        personalized.append((x_hat - X_shared[:, i]) if force_positive else R_hat.flatten())
    return np.column_stack(personalized)


A_personal = decompose_two_shot(A)
S_personal = decompose_two_shot(S, force_positive=True)
V_personal = decompose_two_shot(V)

## 3. Joint clustering on the concatenated personalized components

In [ ]:
min_T = min(A_personal.shape[0], S_personal.shape[0], V_personal.shape[0])
combined_features = np.concatenate([A_personal[:min_T].T, S_personal[:min_T].T, V_personal[:min_T].T], axis=1)

pca_comb = PCA(n_components=5)
X_pca = pca_comb.fit_transform(combined_features)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_pca)
print("Cluster sizes:", np.bincount(labels))

X_2d = PCA(n_components=2).fit_transform(combined_features)
plt.figure(figsize=(10, 10))
sns.scatterplot(x=X_2d[:, 0], y=X_2d[:, 1], hue=labels, palette="Set2", s=120, edgecolor="black")
for i in range(len(labels)):
    plt.text(X_2d[i, 0] + 0.03, X_2d[i, 1], f"V{i + 1}", fontsize=20)
handles, legend_labels = plt.gca().get_legend_handles_labels()
new_labels = [f"Cluster {int(lbl)}" if lbl.isdigit() else lbl for lbl in legend_labels]
plt.xlabel("Personalized Component 1", fontsize=22)
plt.xticks(fontsize=20)
plt.ylabel("Personalized Component 2", fontsize=22)
plt.ylim(-80, 120)
plt.yticks([-50, 0, 50, 100], fontsize=20)
plt.legend(handles, new_labels, fontsize=22, loc="lower center", ncol=3)
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "NGSIM_cluster.png", dpi=300, bbox_inches="tight")
plt.show()